# 🏆 Churn Prediction: Master Production Pipeline (V5.9.2 - FINAL MASTERPIECE)
**Status**: VERSION LOCKED (1.5.1) + FULL ANALYTICS + RAM OPTIMIZED (COMPACT).

In [ ]:
# 1. Setup & Auth (SCIKIT-LEARN LOCKED TO 1.5.1)
!pip install opendatasets mlflow xgboost shap python-dotenv seaborn dagshub scikit-learn==1.5.1 -q

import torch
HAS_GPU = torch.cuda.is_available()

import opendatasets as od
import os, pandas as pd, numpy as np, mlflow, mlflow.sklearn, matplotlib.pyplot as plt, seaborn as sns, shap, dagshub
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import recall_score, classification_report, accuracy_score, precision_score, f1_score
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

mlflow.sklearn.autolog(log_models=False)
dagshub.init(repo_owner="nhannhb92", repo_name="msa24-ddm501-group6-final-project", mlflow=True)

In [ ]:
# 2. Data Loading & Profiling
def clean_dataset(df):
    df.columns = [col.lower().replace(' ', '_') for col in df.columns]
    df = df.drop(columns=[c for c in ['customerid', 'last_interaction'] if c in df.columns])
    if 'age' in df.columns: df = df[(df['age'] >= 18) & (df['age'] <= 120)]
    for col in [c for c in ['total_spend', 'totalcharges'] if c in df.columns]:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df.dropna(subset=['churn'])

if not os.path.exists("customer-churn-dataset"):
    od.download("https://www.kaggle.com/datasets/muhammadshahidazeem/customer-churn-dataset")

df_raw = clean_dataset(pd.read_csv("customer-churn-dataset/customer_churn_dataset-training-master.csv"))
df_test_final = clean_dataset(pd.read_csv("customer-churn-dataset/customer_churn_dataset-testing-master.csv"))
df_train, df_val = train_test_split(df_raw, test_size=0.15, random_state=42, stratify=df_raw['churn'])

print("\n--- 📊 DATA PROFILE ---")
print(f"Splits: Train({len(df_train)}), Val({len(df_val)}), Test({len(df_test_final)})")
print(f"Base Churn Rate: {df_train['churn'].mean():.2%}")

In [ ]:
# 3. Full Visual EDA
plt.figure(figsize=(18, 4))
plt.subplot(1, 4, 1); sns.countplot(x='churn', data=df_train, palette='viridis'); plt.title("Imbalance Check")
plt.subplot(1, 4, 2); df_train.select_dtypes(include=[np.number]).corr()['churn'].sort_values().drop('churn').plot(kind='barh'); plt.title("Correlations")
plt.subplot(1, 4, 3); sns.boxplot(x='churn', y='age', data=df_train, palette='magma'); plt.title("Age vs Churn")
plt.subplot(1, 4, 4); sns.boxplot(x='churn', y='total_spend', data=df_train, palette='magma'); plt.title("Spend vs Churn")
plt.tight_layout(); plt.show()

In [ ]:
# 4. Ultimate Masterpiece Training Engine (Compact)
def run_final_experiment(df_tr, df_va, df_te, model_type):
    X_tr, y_tr = df_tr.drop(columns=['churn']), df_tr['churn']
    X_va, y_va = df_va.drop(columns=['churn']), df_va['churn']
    X_te, y_te = df_te.drop(columns=['churn']), df_te['churn']
    
    pre = ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), X_tr.select_dtypes(include=[np.number]).columns.tolist()),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), X_tr.select_dtypes(include=['object']).columns.tolist())
    ])
    
    pre.fit(X_tr)
    X_va_t = pre.transform(X_va)
    
    if model_type == "xgboost":
        clf = XGBClassifier(scale_pos_weight=(y_tr==0).sum()/(y_tr==1).sum(), device='cuda' if HAS_GPU else 'cpu', early_stopping_rounds=10)
        f_params = {"clf__eval_set": [(X_va_t, y_va)], "clf__verbose": False}
        m_params = {'clf__max_depth': [3, 5], 'clf__n_estimators': [50], 'clf__learning_rate': [0.1]}
    elif model_type == "random_forest":
        clf = RandomForestClassifier(class_weight='balanced', random_state=42)
        f_params = {}; m_params = {'clf__max_depth': [5, 10], 'clf__n_estimators': [50]}
    else:
        clf = LogisticRegression(class_weight='balanced', max_iter=2000)
        f_params = {}; m_params = {'clf__C': [0.1, 1.0]}
        
    mlflow.set_experiment("Churn_Final_Masterpiece")
    with mlflow.start_run(run_name=f"NB_MASTERV59_{model_type.upper()}"):
        pipe = Pipeline([('pre', pre), ('clf', clf)])
        search = RandomizedSearchCV(pipe, m_params, n_iter=2, cv=2, scoring='recall', verbose=1)
        search.fit(X_tr, y_tr, **f_params)
        model = search.best_estimator_
        
        # --- 📈 PERFORMANCE REPORT ---
        y_pred = model.predict(X_te)
        print(f"\n📢 PERFORMANCE SUMMARY ({model_type.upper()}):")
        print(classification_report(y_te, y_pred))
        
        # --- ⚖️ RESPONSIBLE AI AUDIT ---
        res = df_te.copy(); res['pred'] = y_pred
        print(f"\n⚖️ Gender Bias Audit:")
        print(res.groupby('gender').agg(Actual=('churn', 'mean'), Predicted=('pred', 'mean')))
        plt.figure(figsize=(8, 4)); res.groupby('gender')[['churn', 'pred']].mean().plot(kind='bar', ax=plt.gca()); plt.title("Bias Check"); plt.show()
        
        # --- 🔍 EXPLAINABILITY ---
        try:
            sample = X_te.sample(min(150, len(X_te)))
            X_t = pd.DataFrame(model.named_steps['pre'].transform(sample), columns=model.named_steps['pre'].get_feature_names_out())
            explainer = shap.Explainer(model.named_steps['clf'], X_t) if model_type == "logistic_regression" else shap.Explainer(model.named_steps['clf'])
            plt.figure(figsize=(10, 6)); shap.summary_plot(explainer(X_t), X_t, show=False, max_display=15); plt.show()
        except: pass

        from mlflow.models import infer_signature
        mlflow.sklearn.log_model(model, "model", registered_model_name=f"CustomerChurnModel_{model_type}", signature=infer_signature(X_tr, model.predict(X_tr)))
        print(f"✅ SUCCESS: {model_type.upper()} Registered")

for m in ["xgboost", "random_forest", "logistic_regression"]:
    run_final_experiment(df_train, df_val, df_test_final, m)